In [ ]:
from mssm.models import *
from mssmViz.sim import *
from mssmViz.plot import *
from mssm.src.python.compare import compare_CDL
import pickle
import copy
import os
from src.utils import init_lambda

from defaults import (
    default_gamm_kwargs,
    default_gsmm_kwargs,
    default_comparison_kwargs,
)

try:
    os.makedirs("./results")
except:
    print("Result directories already exist.")

try:
    os.makedirs("./results/sim")
    os.makedirs("./results/data")
    os.makedirs("./results/plots")
    os.makedirs("./results/data/sim1")
    os.makedirs("./results/data/sim2")
    os.makedirs("./results/data/sim3")
    os.makedirs("./results/data/sim1/plots")
    os.makedirs("./results/data/sim3/plots")
    os.makedirs("./results/data/sim4")
    os.makedirs("./results/data/sim5")
except:
    print("Result sub-directory already exist.")

n_cores = 8
prior = None

# Newton
newton_kwargs = copy.deepcopy(default_gsmm_kwargs)
newton_kwargs["prefit_grad"] = False

# Uncorrected
uncor_kwargs = copy.deepcopy(default_comparison_kwargs)
uncor_kwargs["correct_V"] = False
uncor_kwargs["correct_t1"] = True
uncor_kwargs["grid"] = "JJJ3"
uncor_kwargs["only_expected_edf"] = False

# WPS PQL like
wps_pql_kwargs = copy.deepcopy(default_comparison_kwargs)
wps_pql_kwargs["correct_t1"] = False
wps_pql_kwargs["grid"] = "JJJ1"
wps_pql_kwargs["only_expected_edf"] = False
wps_pql_kwargs["compute_Vcc"] = True

# Expected
pql_NORM_JJJ2_kwargs = copy.deepcopy(default_comparison_kwargs)
pql_NORM_JJJ2_kwargs["df"] = np.inf
pql_NORM_JJJ2_kwargs["correct_t1"] = False
pql_NORM_JJJ2_kwargs["n_c"] = n_cores
pql_NORM_JJJ2_kwargs["grid"] = "JJJ2"
pql_NORM_JJJ2_kwargs["only_expected_edf"] = False
pql_NORM_JJJ2_kwargs["use_importance_weights"] = False
pql_NORM_JJJ2_kwargs["compute_Vcc"] = True

pql_REML_JJJ3_kwargs = copy.deepcopy(default_comparison_kwargs)
pql_REML_JJJ3_kwargs["correct_t1"] = False
pql_REML_JJJ3_kwargs["n_c"] = n_cores
pql_REML_JJJ3_kwargs["grid"] = "JJJ3"
pql_REML_JJJ3_kwargs["only_expected_edf"] = False
pql_REML_JJJ3_kwargs["use_importance_weights"] = True
pql_REML_JJJ3_kwargs["recompute_H"] = True
pql_REML_JJJ3_kwargs["prior"] = prior
pql_REML_JJJ3_kwargs["compute_Vcc"] = True

In [ ]:
############################# Simulation 4 #############################
n_c = 10
n_sim = 100
df = 40
nR = 250

pql_NORM_JJJ2_kwargs["nR"] = nR
pql_REML_JJJ3_kwargs["nR"] = nR
pql_REML_JJJ3_kwargs["df"] = df

families = ["Gaussian","Binomial","Gamma"]
binom_offsets = [0,-5,0]
scales = [2,2,2]

for family,binom_offset,scale in zip(families,binom_offsets,scales):

    Failures = np.zeros(n_c)
    AIC_rej = np.zeros(n_c)
    AIC_t1_rej = np.zeros(n_c)

    eta_mses = np.zeros((n_c,n_sim,2))

    # WPS-like
    WPS_AIC_PQL_rej = np.zeros(n_c)

    # Expected AICs
    edf_PQL_NORM_JJJ2_AIC_rej  = np.zeros(n_c)
    edf_PQL_REML_JJJ3_AIC_rej  = np.zeros(n_c)

    for c_i,c_val in enumerate(np.linspace(0,1,n_c)):

        iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
        for sim_i in iterator:

            ######################################## Family setup ########################################
            mod_fam = Gaussian()
            if family == "Binomial":
                mod_fam = Binomial()
            if family == "Gamma":
                mod_fam = Gamma()

            ######################################## Create Data ########################################
            sim_fit_dat = sim3(n=500,scale=scale,c=c_val,family=mod_fam,binom_offset=binom_offset,seed=sim_i)
            sim_fit_dat.to_csv(f'./results/data/sim4/sim_size:{n_sim}_c:{c_i}_fam:{family}_nR:{nR}_df:{df}_scale:{scale}_binomoffset:{binom_offset}_set:{sim_i}.csv',index=False)

            ######################################## First Formula + Models ########################################
            sim_fit_formula = Formula(lhs("y"),
                                        [i(),f(["x0"],nk=9),f(["x1"],nk=9),f(["x2"],nk=9),f(["x3"],nk=9)],
                                        data=sim_fit_dat,
                                        print_warn=False)

            sim_fit_model = GAMM(sim_fit_formula,mod_fam)

            sim_fit_model.fit(**default_gamm_kwargs)

            ######################################## Second Formula + Models ########################################
            sim_fit_formula2 = Formula(lhs("y"),
                                        [i(),f(["x1"],nk=9),f(["x2"],nk=9),f(["x3"],nk=9)],
                                            data=sim_fit_dat,
                                            print_warn=False)

            sim_fit_model2 = GAMM(sim_fit_formula2,mod_fam)

            sim_fit_model2.fit(**default_gamm_kwargs)

            try:
                for mi,model in enumerate([sim_fit_model,sim_fit_model2]): #
                    pred_diff = model.preds[0].flatten() - sim_fit_dat["eta"].values
                    eta_mses[c_i,sim_i,mi] = np.dot(pred_diff,pred_diff)/len(pred_diff)
                    #print(eta_mses[c_i,sim_i,:])
            except:
                print("Error collecting MSEs")
            

            ######################################## Comparisons ########################################
            uncor_kwargs["seed"] = sim_i
            wps_pql_kwargs["seed"] = sim_i
            pql_NORM_JJJ2_kwargs["seed"] = sim_i
            pql_REML_JJJ3_kwargs["seed"] = sim_i

            try:
                uncor_result = compare_CDL(sim_fit_model,sim_fit_model2,**uncor_kwargs)
                
                # Compute heuristic upper edf from WPS (2016) as well
                llk1 = uncor_result["aic1"] - 2*uncor_result["DOF12"]
                llk2 = uncor_result["aic2"] - 2*uncor_result["DOF22"] 
                aic1_t1 =  llk1 + 2*uncor_result["DOF1"]
                aic2_t1 =  llk2 + 2*uncor_result["DOF2"]

                aic_diff_t1 = aic1_t1 - aic2_t1

                # WPS:
                wps_pql_result = compare_CDL(sim_fit_model,sim_fit_model2,**wps_pql_kwargs)

                # Expected
                if family == "Gaussian":
                    pql_NORM_JJJ2_edf_result = compare_CDL(sim_fit_model,sim_fit_model2,**pql_NORM_JJJ2_kwargs)
                else:
                    pql_REML_JJJ3_edf_result = compare_CDL(sim_fit_model,sim_fit_model2,**pql_REML_JJJ3_kwargs)

                if uncor_result["aic_diff"] < 0:
                    AIC_rej[c_i] += 1
                if aic_diff_t1 < 0:
                    AIC_t1_rej[c_i] += 1
                
                # WPS
                if wps_pql_result["aic_diff"] < 0:
                    WPS_AIC_PQL_rej[c_i] += 1

                # Expected:
                if family == "Gaussian" and pql_NORM_JJJ2_edf_result["aic_diff"] < 0:
                    edf_PQL_NORM_JJJ2_AIC_rej[c_i] += 1
                if family != "Gaussian" and pql_REML_JJJ3_edf_result["aic_diff"] < 0:
                    edf_PQL_REML_JJJ3_AIC_rej[c_i] += 1


            except:
                print(f"Failure for family: {family} at sim: {[sim_i,c_i]} in compare code")
                Failures[c_i] += 1

            # Show some progress info
            progress = [np.round(AIC_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2),
                        np.round(WPS_AIC_PQL_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2),
                        np.round(edf_PQL_REML_JJJ3_AIC_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2),
                        np.round(edf_PQL_NORM_JJJ2_AIC_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2)]

            iterator.set_description_str(desc=f"Acc (UC, WPS, UpperR, UpperN).: {progress}", refresh=True)

        iterator.close()
    
    ######################################## Save & Plot ########################################
    print(Failures)
    completed_sim = (n_sim - Failures)

    AIC_rej /= completed_sim
    AIC_t1_rej /= completed_sim
    WPS_AIC_PQL_rej /= completed_sim
    edf_PQL_NORM_JJJ2_AIC_rej /= completed_sim
    edf_PQL_REML_JJJ3_AIC_rej /= completed_sim

    sim_outcome = {"AIC_rej":AIC_rej,
                    "AIC_t1_rej":AIC_t1_rej,
                    "WPS_AIC_PQL_rej":WPS_AIC_PQL_rej,
                    "edf_PQL_NORM_JJJ2_AIC_rej":edf_PQL_NORM_JJJ2_AIC_rej,
                    "edf_PQL_REML_JJJ3_AIC_rej":edf_PQL_REML_JJJ3_AIC_rej,
                    "Failures":Failures,
                    "MSE":eta_mses}

    with open(f'./results/sim/sim_4_size:{n_sim}_n_c:{n_c}_fam:{family}_nR:{nR}_df:{df}_scale:{scale}_binomoffset:{binom_offset}.pickle', 'wb') as file:
        pickle.dump(sim_outcome,file, protocol=pickle.HIGHEST_PROTOCOL)

    plt.plot(np.linspace(0,1,n_c),AIC_rej,label='AIC')
    plt.plot(np.linspace(0,1,n_c),AIC_t1_rej,label='AIC t1')
    plt.plot(np.linspace(0,1,n_c),WPS_AIC_PQL_rej,label='AIC WPS PQL')
    plt.plot(np.linspace(0,1,n_c),edf_PQL_NORM_JJJ2_AIC_rej,label='AIC NORM JJJ2')
    plt.plot(np.linspace(0,1,n_c),edf_PQL_REML_JJJ3_AIC_rej,label='AIC REML JJJ3')
    plt.legend()
    plt.show()


In [ ]:
############################# Simulation 4 #############################
n_c = 10
n_sim = 100
df = 40
nR = 250
scale = 2

pql_REML_JJJ3_kwargs["nR"] = nR
pql_REML_JJJ3_kwargs["df"] = df


Failures = np.zeros(n_c)
AIC_rej = np.zeros(n_c)
AIC_t1_rej = np.zeros(n_c)

eta_mses = np.zeros((n_c,n_sim,2))

# WPS-like
WPS_AIC_PQL_rej = np.zeros(n_c)

# Expected AICs
edf_PQL_REML_JJJ3_AIC_rej  = np.zeros(n_c)

for c_i,c_val in enumerate(np.linspace(0,1,n_c)):


    iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
    for sim_i in iterator:

        ######################################## Create Data ########################################
        sim_fit_dat = sim3(n=500,scale=scale,c=c_val,family=PropHaz([0],[0]),binom_offset=0.1,seed=sim_i)
        sim_fit_dat = sim_fit_dat.sort_values(['y'],ascending=[False])
        sim_fit_dat = sim_fit_dat.reset_index(drop=True)

        u,inv = np.unique(sim_fit_dat["y"],return_inverse=True)
        ut = np.flip(u)
        r = np.abs(inv - max(inv))
        
        sim_fit_dat.to_csv(f'./results/data/sim4/sim_size:{n_sim}_c:{c_i}_fam:{"PropHaz"}_nR:{nR}_df:{df}_scale:{scale}_set:{sim_i}.csv',index=False)

        ######################################## First Formula + Models ########################################
        sim_fit_formula = Formula(lhs("delta"),
                                        [f(["x0"],nk=9),f(["x1"],nk=9),f(["x2"],nk=9),f(["x3"],nk=9)],
                                        data=sim_fit_dat,
                                        print_warn=False)
        
        sim_formulas = [sim_fit_formula]

        gsmm_newton_fam = PropHaz(ut,r)
        gsmm_newton_fam.init_lambda = init_lambda
        sim_fit_model = GSMM(copy.deepcopy(sim_formulas),gsmm_newton_fam)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            newton_kwargs["seed"] = sim_i
            sim_fit_model.fit(**newton_kwargs)

        ######################################## Second Formula + Models ########################################
        sim_fit_formula2 = Formula(lhs("delta"),
                                        [f(["x1"],nk=9),f(["x2"],nk=9),f(["x3"],nk=9)],
                                            data=sim_fit_dat,
                                            print_warn=False)
        
        sim_formulas2 = [sim_fit_formula2]

        gsmm_newton_fam2 = PropHaz(ut,r)
        gsmm_newton_fam2.init_lambda = init_lambda
        sim_fit_model2 = GSMM(copy.deepcopy(sim_formulas2),gsmm_newton_fam2)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            newton_kwargs["seed"] = sim_i
            sim_fit_model2.fit(**newton_kwargs)

        try:
            for mi,model in enumerate([sim_fit_model,sim_fit_model2]): #
                pred_diff = model.preds[0].flatten() - sim_fit_dat["eta"].values
                eta_mses[c_i,sim_i,mi] = np.dot(pred_diff,pred_diff)/len(pred_diff)
                
        except:
            print("Error collecting MSEs")
        
        ######################################## Comparisons ########################################
        uncor_kwargs["seed"] = sim_i
        wps_pql_kwargs["seed"] = sim_i
        pql_REML_JJJ3_kwargs["seed"] = sim_i

        try:
            uncor_result = compare_CDL(sim_fit_model,sim_fit_model2,**uncor_kwargs)
            
            # Compute heuristic upper edf from WPS (2016) as well
            llk1 = uncor_result["aic1"] - 2*uncor_result["DOF12"]
            llk2 = uncor_result["aic2"] - 2*uncor_result["DOF22"] 
            aic1_t1 =  llk1 + 2*uncor_result["DOF1"]
            aic2_t1 =  llk2 + 2*uncor_result["DOF2"]

            aic_diff_t1 = aic1_t1 - aic2_t1

            # WPS:
            wps_pql_result = compare_CDL(sim_fit_model,sim_fit_model2,**wps_pql_kwargs)

            # Expected
            pql_REML_JJJ3_edf_result = compare_CDL(sim_fit_model,sim_fit_model2,**pql_REML_JJJ3_kwargs)

            if uncor_result["aic_diff"] < 0:
                AIC_rej[c_i] += 1
            if aic_diff_t1 < 0:
                AIC_t1_rej[c_i] += 1
            
            # WPS
            if wps_pql_result["aic_diff"] < 0:
                WPS_AIC_PQL_rej[c_i] += 1

            # Expected:
            if pql_REML_JJJ3_edf_result["aic_diff"] < 0:
                edf_PQL_REML_JJJ3_AIC_rej[c_i] += 1


        except:
            print(f"Failure for family: {"PropHaz"} at sim: {[sim_i,c_i]} in compare code")
            Failures[c_i] += 1

        # Show some progress info
        progress = [np.round(AIC_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2),
                    np.round(WPS_AIC_PQL_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2),
                    np.round(edf_PQL_REML_JJJ3_AIC_rej[c_i]/(sim_i + 1 - Failures[c_i]),decimals=2)]

        iterator.set_description_str(desc=f"Acc (UC, WPS, UpperR).: {progress}", refresh=True)

    iterator.close()

######################################## Save & Plot ########################################
print(Failures)
completed_sim = (n_sim - Failures)

AIC_rej /= completed_sim
AIC_t1_rej /= completed_sim
WPS_AIC_PQL_rej /= completed_sim
edf_PQL_REML_JJJ3_AIC_rej /= completed_sim


sim_outcome = {"AIC_rej":AIC_rej,
                "AIC_t1_rej":AIC_t1_rej,
                "WPS_AIC_PQL_rej":WPS_AIC_PQL_rej,
                "edf_PQL_REML_JJJ3_AIC_rej":edf_PQL_REML_JJJ3_AIC_rej,
                "Failures":Failures,
                "MSE":eta_mses}

with open(f'./results/sim/sim_4_size:{n_sim}_n_c:{n_c}_fam:{"PropHaz"}_nR:{nR}_df:{df}_scale:{scale}.pickle', 'wb') as file:
    pickle.dump(sim_outcome,file, protocol=pickle.HIGHEST_PROTOCOL)

plt.plot(np.linspace(0,1,n_c),AIC_rej,label='AIC')
plt.plot(np.linspace(0,1,n_c),AIC_t1_rej,label='AIC t1')
plt.plot(np.linspace(0,1,n_c),WPS_AIC_PQL_rej,label='AIC WPS PQL')
plt.plot(np.linspace(0,1,n_c),edf_PQL_REML_JJJ3_AIC_rej,label='AIC REML JJJ3')
plt.legend()
plt.show()
